In [1]:
import json
import time
import re
from dataclasses import dataclass, field
from urllib.parse import urlparse, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException

# 0. Exploration manuelle : Sephora

Avant d'écrire les fonctions générales des sections suivantes (celles qui tourneront sur toute une liste de sites d'un coup), on explore **une seule page, à la main**, feature par feature — comme si on découvrait sa structure pour la première fois. C'est exactement cette exploration qui a guidé la conception de `find_button_across_frames`, `classify_cookies`, `read_tracker_domains_from_perf_log`, etc. dans les sections suivantes.

Cible choisie : **sephora.co.uk**. `sephora.fr` était injoignable (bloqué par la protection anti-bot Akamai) depuis l'environnement où ce notebook a été rédigé ; la version UK répond, elle, et affiche un vrai bandeau cookies (CMP OneTrust), ce qui permet de dérouler l'exploration à l'identique.

In [2]:
TARGET_URL = "https://www.sephora.co.uk"

# Etape 1 : le site repond-il a une simple requete HTTP, sans navigateur ?
resp = requests.get(TARGET_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
print(resp.status_code, "-", len(resp.text), "caracteres recus")

200 - 584168 caracteres recus


Ca repond (200) et on recoit du HTML. Peut-on deja deviner le CMP (l'outil de gestion du consentement cookies) rien qu'en fouillant ce HTML brut, sans executer le moindre Javascript ?

In [3]:
# Etape 2 : chercher la signature d'un CMP connu dans le HTML brut
html_lower = resp.text.lower()
for cmp_name in ["onetrust", "optanon", "didomi", "cookiebot", "trustarc", "usercentrics"]:
    if cmp_name in html_lower:
        print(f"trouve dans le HTML brut : {cmp_name}")

trouve dans le HTML brut : onetrust
trouve dans le HTML brut : optanon


Le CMP (OneTrust) est repere : son script est reference directement dans le HTML servi par le serveur, avant meme toute execution JS. Est-ce qu'on retrouve aussi le texte du bandeau ("Refuser", "Accepter"...) de la meme facon ? Cherchons avec BeautifulSoup, sur tous les boutons et liens de la page.

In [4]:
# Etape 3 : chercher le bandeau cookies dans le HTML statique
soup_static = BeautifulSoup(resp.text, "html.parser")
elements_static = [el.get_text(strip=True) for el in soup_static.find_all(["button", "a"])]
cookie_matches_static = [t for t in elements_static if t and any(k in t.lower() for k in ["refuser", "accepter", "cookie"])]

print(f"{len(elements_static)} boutons/liens trouves au total, {len(cookie_matches_static)} contiennent un mot-cle cookie :")
print(cookie_matches_static)

758 boutons/liens trouves au total, 1 contiennent un mot-cle cookie :
['Privacy & Cookies']


Le seul match, c'est le lien "Privacy & Cookies" du pied de page — pas le bandeau lui-meme. Les boutons "Refuser" / "Accepter" n'existent tout simplement pas encore dans ce HTML : le CMP les injecte via Javascript, apres coup, une fois la page chargee dans un navigateur. `requests` + `BeautifulSoup` ne peuvent pas voir ca, il faut un vrai navigateur : direction Selenium.

On active tout de suite les logs de performance Chrome (`goog:loggingPrefs`) : c'est ce qui permettra plus loin de lire les requetes reseau (cookies tiers, trackers). Les deux options `excludeSwitches` / `useAutomationExtension` retirent simplement la banniere "Chrome is being controlled by automated software" — une config Selenium standard, pas une technique d'evasion.

In [5]:
# Etape 4 : ouvrir la page dans un vrai navigateur (headless Chrome)
manual_options = Options()
manual_options.add_argument("--headless=new")
manual_options.add_argument("--window-size=1400,1000")
manual_options.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15"
)
manual_options.add_experimental_option("excludeSwitches", ["enable-automation"])
manual_options.add_experimental_option("useAutomationExtension", False)
manual_options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

manual_driver = webdriver.Chrome(options=manual_options)
manual_driver.get(TARGET_URL)
time.sleep(5)  # laisser le CMP + les scripts tiers se charger

print(manual_driver.title)
print(len(manual_driver.page_source), "caracteres de HTML rendu (contre", len(resp.text), "en HTML brut)")

SEPHORA UK | Beauty, Makeup, Skincare, Haircare & Fragrances
758262 caracteres de HTML rendu (contre 584168 en HTML brut)


Beaucoup plus de HTML rendu qu'en brut : le Javascript a bien tourne. On refait la meme recherche de boutons, mais cette fois dans le DOM rendu par le navigateur.

In [6]:
# Etape 5 : chercher le bandeau dans le DOM rendu (document principal)
buttons_rendered = manual_driver.find_elements(
    "css selector", "button, a[role='button'], [role='button']"
)
texts_rendered = [b.text.strip() for b in buttons_rendered if b.text.strip()]
cookie_matches_rendered = [t for t in texts_rendered if any(k in t.lower() for k in ["refuser", "accepter", "autoriser", "cookie", "paramet"])]

print(f"{len(texts_rendered)} boutons avec du texte, {len(cookie_matches_rendered)} lies au bandeau cookies :")
print(cookie_matches_rendered)

14 boutons avec du texte, 3 lies au bandeau cookies :
['Tout refuser', 'Autoriser tous les cookies', 'Paramètres des cookies']


Le bandeau est bien la, avec un bouton "Tout refuser" qui matche exactement le mot-cle "refuser" auquel on s'attendait. Mais est-ce que la liste de mots-cles "accepter" qu'on avait en tete au depart (`"autoriser tout"`) aurait vraiment matche le texte reel du bouton ?

In [7]:
# Etape 6 : confronter le texte reel du bouton a notre hypothese de mot-cle
accept_button_text = next(t for t in cookie_matches_rendered if "autoris" in t.lower())
naive_keyword = "autoriser tout"

print(f"texte reel du bouton     : {accept_button_text!r}")
print(f"mot-cle imagine au depart : {naive_keyword!r}")
print("le mot-cle matche le texte reel ?", naive_keyword in accept_button_text.lower())

texte reel du bouton     : 'Autoriser tous les cookies'
mot-cle imagine au depart : 'autoriser tout'
le mot-cle matche le texte reel ? False


"autoriser tout" (singulier) ne matche pas "autoriser **tous** les cookies" (pluriel) : un ecart discret mais reel entre ce qu'on imagine et ce que le site affiche vraiment. C'est exactement ce genre de decouverte, site par site, qui a fait grossir les listes `ACCEPT_KEYWORDS` / `CUSTOMIZE_KEYWORDS` de la Section 1.

On continue l'exploration : combien de cookies sont deja poses **avant** qu'on ait cliqué quoi que ce soit ?

In [8]:
# Etape 7 : cookies poses AVANT toute interaction avec le bandeau
cookies_before = manual_driver.get_cookies()
print(f"{len(cookies_before)} cookies avant interaction :")
print(sorted(c["name"] for c in cookies_before))

13 cookies avant interaction :
['AKA_A2', 'OptanonConsent', 'PHPSESSID', 'RT', '__blka_ts', '_abck', '_blkp_xps', 'ak_bmsc', 'bm_sz', 'feeluniqueCurr', 'feeluniqueLoc', 'optin_location', 'selected_store']


Et cote reseau : est-ce que des requetes partent deja vers des domaines tiers (pubs, analytics...) avant meme d'avoir donne son consentement ? Les cookies ne montrent que la moitie de l'histoire — il faut regarder les vraies requetes HTTP, via les logs de performance Chrome qu'on a active a l'etape 4.

In [9]:
# Etape 8 : requetes tierces AVANT interaction, lues dans le performance log
perf_logs_before = manual_driver.get_log("performance")  # chaque appel vide le buffer

main_domain = urlparse(TARGET_URL).netloc.replace("www.", "")
third_party_before = set()
for entry in perf_logs_before:
    message = json.loads(entry["message"])["message"]
    if message.get("method") == "Network.requestWillBeSent":
        req_domain = urlparse(message["params"]["request"]["url"]).netloc
        if req_domain and main_domain not in req_domain:
            third_party_before.add(req_domain)

print(f"{len(third_party_before)} domaines tiers contactes avant interaction :")
print(sorted(third_party_before))

25 domaines tiers contactes avant interaction :
['02179915.akstat.io', 'ade.googlesyndication.com', 'assets.feelunique.com', 'bam.nr-data.net', 'c.go-mpulse.net', 'cdn-eu.dynamicyield.com', 'cdn-ukwest.onetrust.com', 'cdn.attraqt.io', 'cdn.jsdelivr.net', 'cdn1.feelunique.com', 'collect-eu.attraqt.io', 'd2l3syy27ktbfg.cloudfront.net', 'geolocation.onetrust.com', 'js-agent.newrelic.com', 'pagead2.googlesyndication.com', 'portal.brandlock.io', 'rcom-eu.dynamicyield.com', 'region1.google-analytics.com', 'rum-collector-2.pingdom.net', 'rum-static.pingdom.net', 's2.go-mpulse.net', 'st-eu.dynamicyield.com', 'unpkg.com', 'websdk.appsflyer.com', 'www.googletagmanager.com']


Deja des domaines tiers actifs avant tout clic. Test le plus important de l'audit CNIL : est-ce que cliquer sur "Refuser" **change reellement quelque chose**, ou est-ce un bouton de facade ?

In [10]:
# Etape 9 : cliquer sur "Tout refuser" et remesurer
reject_button = next(
    b for b in manual_driver.find_elements("css selector", "button, a[role='button'], [role='button']")
    if b.text.strip().lower() == "tout refuser"
)
manual_driver.execute_script("arguments[0].click();", reject_button)
time.sleep(3)  # laisser partir une eventuelle requete tierce

cookies_after = manual_driver.get_cookies()
perf_logs_after = manual_driver.get_log("performance")
third_party_after = set()
for entry in perf_logs_after:
    message = json.loads(entry["message"])["message"]
    if message.get("method") == "Network.requestWillBeSent":
        req_domain = urlparse(message["params"]["request"]["url"]).netloc
        if req_domain and main_domain not in req_domain:
            third_party_after.add(req_domain)

new_cookie_names = sorted({c["name"] for c in cookies_after} - {c["name"] for c in cookies_before})

print(f"cookies                        : {len(cookies_before)} avant -> {len(cookies_after)} apres")
print(f"nouveaux cookies apparus       : {new_cookie_names or '(aucun)'}")
print(f"domaines tiers (fenetre de 3s) : {len(third_party_before)} avant -> {len(third_party_after)} apres")

cookies                        : 13 avant -> 14 apres
nouveaux cookies apparus       : ['OptanonAlertBoxClosed']
domaines tiers (fenetre de 3s) : 25 avant -> 2 apres


Derniere feature a explorer : le lien vers la politique de confidentialite, sur la page maintenant rendue (post-clic).

In [11]:
# Etape 10 : lien vers la politique de confidentialite
soup_rendered = BeautifulSoup(manual_driver.page_source, "html.parser")
policy_link = None
for link in soup_rendered.find_all("a", href=True):
    link_text = link.get_text(strip=True)
    if "privacy" in link_text.lower() or "cookie" in link_text.lower():
        policy_link = (link_text, urljoin(TARGET_URL, link["href"]))
        break

print("lien trouve :", policy_link)

# Nos mots-cles "formels" (POLICY_LINK_WORDS, Section 6) attendent des phrases
# completes comme "privacy policy" -- le texte reel ici est "Privacy & Cookies",
# qui ne matche AUCUNE de ces phrases malgre le mot "privacy" au milieu.
formal_keyword_match = "privacy policy" in policy_link[0].lower() if policy_link else False
print(f"aurait matche la liste formelle POLICY_LINK_WORDS ? {formal_keyword_match}")

lien trouve : ('Privacy & Cookies', 'https://www.sephora.co.uk/privacy-policy')
aurait matche la liste formelle POLICY_LINK_WORDS ? False


Encore un ecart entre l'attendu et le reel : "Privacy & Cookies" ne matche pas la phrase exacte "privacy policy". Derniere case a cocher : robots.txt, un simple GET sur un chemin fixe, aucune interaction requise.

In [12]:
# Etape 11 : robots.txt, puis on ferme le navigateur
manual_driver.get(f"{urlparse(TARGET_URL).scheme}://{urlparse(TARGET_URL).netloc}/robots.txt")
time.sleep(1)
print("robots.txt present :", "user-agent" in manual_driver.page_source.lower())

manual_driver.quit()

robots.txt present : True


## Bilan de l'exploration manuelle

En suivant Sephora pas a pas, on a touche du doigt toutes les features qu'on veut mesurer, et pourquoi chacune demande la technique qu'elle demande :

| Etape manuelle | Ce qu'on a decouvert | Devient, en Section 1-8 |
|---|---|---|
| 2. Signature CMP dans le HTML brut | detectable sans navigateur | logique reprise dans `analyze_site` (§7.2) |
| 3-4. Bandeau absent du HTML brut | injecte en JS -> Selenium obligatoire | `build_driver` (§3) |
| 5-6. Recherche de boutons + ecart de mot-cle | "autoriser tout" != "autoriser tous" | `ACCEPT_KEYWORDS` / `CUSTOMIZE_KEYWORDS` (§1), `find_button_across_frames` (§4) |
| 7-8. Cookies + requetes tierces avant clic | des trackers actifs des le chargement | `classify_cookies`, `read_tracker_domains_from_perf_log` (§5) |
| 9. Avant/apres le clic sur "Refuser" | test differentiel = coeur de l'audit CNIL | bloc "test differentiel" de `analyze_site` (§7.5) |
| 10. Lien politique de confidentialite | "Privacy & Cookies" != "privacy policy" | `find_policy_link`, `POLICY_LINK_WORDS` (§6) |
| 11. robots.txt | simple GET, pas besoin d'interaction | fin de `analyze_site` (§7.7) |

Deux ecarts reels trouves ici ("autoriser tous" et "privacy & cookies") ont ete ajoutes aux listes de mots-cles de la Section 1 : c'est concretement comme ca que les fonctions generales des sections suivantes s'affinent, exploration apres exploration, site apres site. Le reste du notebook applique ces memes idees a une liste de sites (`SITES`), au lieu d'une seule URL testee a la main.

# 1. Config

In [13]:
SITES = [
    "https://www.lemonde.fr",
    "https://www.leboncoin.fr",
]

# Listes multilingues (FR, EN, DE, ES, IT, NL, PT) — nécessaire pour un
# scraping à l'échelle européenne, une bannière en allemand ou en
# italien ne matchera jamais des keywords uniquement FR/EN.
ACCEPT_KEYWORDS = [
    # Français
    "tout accepter", "accepter tout", "j'accepte", "accepter",
    "autoriser tout", "autoriser tous", "j'autorise",
    # Anglais
    "accept all", "i accept", "accept cookies", "allow all",
    # Allemand
    "alle akzeptieren", "akzeptieren", "zustimmen", "ich stimme zu",
    # Espagnol
    "aceptar todo", "aceptar todas", "aceptar",
    # Italien
    "accetta tutto", "accetta tutti", "accetta",
    # Néerlandais
    "alles accepteren", "accepteren",
    # Portugais
    "aceitar tudo", "aceitar todos", "aceitar",
]
REJECT_KEYWORDS = [
    # Français
    "tout refuser", "refuser tout", "je refuse", "refuser",
    "continuer sans accepter", "je décline",
    # Anglais
    "reject all", "decline", "reject", "decline all", "deny all",
    # Allemand
    "alle ablehnen", "ablehnen", "ich lehne ab",
    # Espagnol
    "rechazar todo", "rechazar todas", "rechazar",
    # Italien
    "rifiuta tutto", "rifiuta tutti", "rifiuta",
    # Néerlandais
    "alles weigeren", "weigeren",
    # Portugais
    "rejeitar tudo", "rejeitar todos", "rejeitar",
]
CUSTOMIZE_KEYWORDS = [
    # Français
    "personnaliser", "paramétrer", "gérer mes choix", "gérer les cookies",
    "préférences",
    # Anglais
    "customize", "manage", "manage preferences", "cookie settings",
    "manage cookies",
    # Allemand
    "einstellungen", "anpassen", "cookie-einstellungen",
    # Espagnol
    "personalizar", "configurar", "ajustes de cookies",
    # Italien
    "personalizza", "gestisci preferenze", "impostazioni cookie",
    # Néerlandais
    "voorkeuren", "cookie-instellingen", "aanpassen",
    # Portugais
    "personalizar", "gerir preferências", "definições de cookies",
]

KNOWN_CMP_SIGNATURES = {
    "didomi": ["didomi", "didomiState"],
    "onetrust": ["onetrust", "optanon"],
    "axeptio": ["axeptio", "axeptioSettings"],
    "cookiebot": ["cookiebot", "cookieconsent"],
    "usercentrics": ["usercentrics"],
    "trustarc": ["trustarc"],
    "sourcepoint": ["sp_message_iframe", "sourcepoint"],
    "funding_choices": ["fundingchoices", "funding choices"],
}

# Fallback si disconnect_services.json n'est pas fourni
FALLBACK_TRACKER_DOMAINS = [
    "google-analytics.com", "googletagmanager.com", "doubleclick.net",
    "facebook.net", "facebook.com", "hotjar.com", "criteo.com",
    "adnxs.com", "amazon-adsystem.com", "adsrvr.org", "taboola.com",
    "outbrain.com", "pubmatic.com", "rubiconproject.com",
]

DISCONNECT_LIST_PATH = "../raw_data/disconnect_services.json"

POLICY_LINK_WORDS = [
    # Anglais
    "privacy policy", "privacy notice", "cookie policy", "privacy & cookies",
    # Français
    "politique de confidentialite", "politique de confidentialité",
    "mentions legales", "mentions légales", "politique de cookies",
    # Allemand
    "datenschutz", "datenschutzerklärung", "cookie-richtlinie",
    # Espagnol
    "politica de privacidad", "política de privacidad",
    "aviso de privacidad", "politica de cookies",
    # Italien
    "informativa sulla privacy", "politica sulla privacy",
    "politica sui cookie",
    # Néerlandais
    "privacybeleid", "privacyverklaring", "cookiebeleid",
    # Portugais
    "politica de privacidade", "política de privacidade",
    "politica de cookies",
]



In [14]:


def load_tracker_domain_map():
    """Charge la liste Disconnect.me si présente localement, sinon fallback."""
    try:
        with open(DISCONNECT_LIST_PATH, "r") as f:
            data = json.load(f)
        domain_map = {}
        for category, services in data["categories"].items():
            for service in services:
                for company, urls_dict in service.items():
                    for homepage, domains in urls_dict.items():
                        for d in domains:
                            domain_map[d] = category
        print(f"[i] {len(domain_map)} domaines trackers chargés (Disconnect.me)")
        return domain_map
    except Exception:
        print("[i] disconnect_services.json non trouvé -> liste fallback utilisée")
        return {d: "Advertising" for d in FALLBACK_TRACKER_DOMAINS}


TRACKER_DOMAIN_MAP = load_tracker_domain_map()


def is_known_tracker(domain: str) -> bool:
    for known_domain, category in TRACKER_DOMAIN_MAP.items():
        if known_domain in domain and category in ("Advertising", "Analytics"):
            return True
    return False

[i] 4443 domaines trackers chargés (Disconnect.me)


# 2. Structure de résultat

In [15]:
@dataclass
class SiteAudit:
    url: str
    reachable: bool = False
    error: str = ""

    # Bannière / boutons (détection cross-iframe)
    has_accept_button: bool = False
    has_reject_button: bool = False
    has_customize_option: bool = False
    has_cookie_banner: bool = False
    reject_as_visible_as_accept: bool = False
    prechecked_boxes_detected: bool = False

    cmp_detected: str = ""
    tcf_api_present: bool = False

    # Cookies AVANT toute interaction
    cookies_before_interaction: int = 0
    non_essential_cookies_before_interaction: int = 0
    long_lived_cookies_count: int = 0

    # Requêtes tierces AVANT interaction
    trackers_before_consent: int = 0
    known_tracker_domains_before_consent: list = field(default_factory=list)

    # Test différentiel : APRES avoir cliqué sur "Refuser"
    reject_click_attempted: bool = False
    reject_click_succeeded: bool = False
    cookies_after_reject: int = 0
    non_essential_cookies_after_reject: int = 0
    trackers_after_reject: int = 0
    known_tracker_domains_after_reject: list = field(default_factory=list)

    # Politique de confidentialité
    privacy_policy_link_found: bool = False

    robots_txt_present: bool = False


# 3. Driver Selenium

In [16]:
def build_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1400,1000")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(20)
    return driver


# 4. Helpers cross-iframe

In [17]:
def find_button_across_frames(driver, keywords):
    """
    Cherche un élément cliquable dont le texte matche un des `keywords`,
    dans le document principal PUIS dans chaque iframe de premier niveau.

    IMPORTANT : si un élément est trouvé, le driver reste "switché" dans
    le contexte (main ou iframe) où il se trouve, pour permettre de le
    cliquer juste après. Toujours appeler driver.switch_to.default_content()
    une fois l'action terminée.
    """
    driver.switch_to.default_content()
    try:
        n_frames = len(driver.find_elements("tag name", "iframe"))
    except WebDriverException:
        n_frames = 0

    contexts = [None] + list(range(n_frames))
    for ctx in contexts:
        driver.switch_to.default_content()
        if ctx is not None:
            try:
                driver.switch_to.frame(ctx)
            except WebDriverException:
                continue
        try:
            clickable = driver.find_elements(
                "css selector",
                "button, a[role='button'], [role='button'], input[type='button']",
            )
            for el in clickable:
                try:
                    txt = el.text.strip().lower()
                    if txt and any(k in txt for k in keywords):
                        return el  # driver reste dans le bon contexte
                except Exception:
                    continue
        except WebDriverException:
            continue
    driver.switch_to.default_content()
    return None


def click_element_safely(driver, element) -> bool:
    try:
        driver.execute_script("arguments[0].click();", element)
        return True
    except Exception:
        try:
            element.click()
            return True
        except Exception:
            return False


# 5. Collecte cookies / réseau


In [18]:
ESSENTIAL_NAME_HINTS = {"cf_", "session", "phpsessid", "csrf", "__cf"}


def classify_cookies(cookies):
    """Retourne (total, non_essentiels, longue_duree>13mois)."""
    non_essential, long_lived = 0, 0
    for c in cookies:
        name_l = c.get("name", "").lower()
        if not any(h in name_l for h in ESSENTIAL_NAME_HINTS):
            non_essential += 1
        expiry = c.get("expiry")
        if expiry and (expiry - time.time()) / 86400 > 13 * 30:
            long_lived += 1
    return len(cookies), non_essential, long_lived


def read_tracker_domains_from_perf_log(driver, main_domain):
    """
    Lit le performance log (Chrome DevTools) accumulé DEPUIS le dernier
    appel à get_log() -> chaque appel vide le buffer, donc deux appels
    successifs donnent bien deux fenêtres temporelles distinctes
    (avant / après clic).
    """
    try:
        logs = driver.get_log("performance")
    except WebDriverException:
        return set(), set()

    third_party_domains = set()
    tracker_domains = set()
    for entry in logs:
        try:
            msg = json.loads(entry["message"])["message"]
            if msg.get("method") == "Network.requestWillBeSent":
                req_url = msg["params"]["request"]["url"]
                req_domain = urlparse(req_url).netloc
                if req_domain and main_domain not in req_domain:
                    third_party_domains.add(req_domain)
                    if is_known_tracker(req_domain):
                        tracker_domains.add(req_domain)
        except Exception:
            continue
    return third_party_domains, tracker_domains

# 6. Politique de confidentialité


In [19]:
def find_policy_link(driver, page_url):
    try:
        html = driver.page_source
    except Exception:
        return None
    soup = BeautifulSoup(html, "html.parser")
    for link in soup.find_all("a", href=True):
        text = link.get_text().lower()
        if any(word in text for word in POLICY_LINK_WORDS):
            return urljoin(page_url, link["href"])
    return None


# 7. Analyse d'un site


In [20]:
def analyze_site(driver, url: str) -> SiteAudit:
    audit = SiteAudit(url=url)
    domain = urlparse(url).netloc.replace("www.", "")

    try:
        driver.get(url)
        time.sleep(4)  # laisser le CMP + les trackers se charger
    except WebDriverException as e:
        audit.error = str(e)
        return audit
    audit.reachable = True

    # --- 7.1 Etat AVANT toute interaction ---------------------------
    cookies_before = driver.get_cookies()
    (audit.cookies_before_interaction,
     audit.non_essential_cookies_before_interaction,
     audit.long_lived_cookies_count) = classify_cookies(cookies_before)

    third_party_before, trackers_before = read_tracker_domains_from_perf_log(
        driver, domain
    )
    audit.trackers_before_consent = len(trackers_before)
    audit.known_tracker_domains_before_consent = sorted(trackers_before)

    # --- 7.2 CMP + TCF -------------------------------------------------
    page_source_lower = driver.page_source.lower()
    for cmp_name, signatures in KNOWN_CMP_SIGNATURES.items():
        if any(sig.lower() in page_source_lower for sig in signatures):
            audit.cmp_detected = cmp_name
            break
    try:
        audit.tcf_api_present = bool(
            driver.execute_script("return typeof window.__tcfapi === 'function';")
        )
    except WebDriverException:
        pass

    # --- 7.3 Recherche des boutons (cross-iframe) --------------------
    accept_el = find_button_across_frames(driver, ACCEPT_KEYWORDS)
    accept_size = accept_el.size if accept_el else None
    driver.switch_to.default_content()

    reject_el = find_button_across_frames(driver, REJECT_KEYWORDS)
    reject_size = reject_el.size if reject_el else None
    reject_frame_found_in = reject_el is not None
    driver.switch_to.default_content()

    customize_el = find_button_across_frames(driver, CUSTOMIZE_KEYWORDS)
    driver.switch_to.default_content()

    audit.has_accept_button = accept_el is not None
    audit.has_reject_button = reject_el is not None
    audit.has_customize_option = customize_el is not None
    audit.has_cookie_banner = audit.has_accept_button or audit.has_reject_button

    if accept_size and reject_size:
        a = accept_size.get("width", 0) * accept_size.get("height", 0)
        r = reject_size.get("width", 0) * reject_size.get("height", 0)
        audit.reject_as_visible_as_accept = (min(a, r) / max(a, r) >= 0.7) if max(a, r) > 0 else False

    # --- 7.4 Cases pré-cochées ---------------------------------------
    try:
        driver.switch_to.default_content()
        checkboxes = driver.find_elements("css selector", "input[type='checkbox']")
        audit.prechecked_boxes_detected = any(
            cb.is_selected() for cb in checkboxes if cb.is_displayed()
        )
    except WebDriverException:
        pass
    driver.switch_to.default_content()

    # --- 7.5 TEST DIFFERENTIEL : clic sur Refuser ---------------------
    if reject_el is not None:
        audit.reject_click_attempted = True
        # find_button_across_frames a laissé le driver dans le bon frame
        reject_el_fresh = find_button_across_frames(driver, REJECT_KEYWORDS)
        if reject_el_fresh is not None:
            audit.reject_click_succeeded = click_element_safely(driver, reject_el_fresh)
        driver.switch_to.default_content()
        time.sleep(3)  # laisser une éventuelle requête tierce partir

        cookies_after = driver.get_cookies()
        (total_after, non_ess_after, _) = classify_cookies(cookies_after)
        audit.cookies_after_reject = total_after
        audit.non_essential_cookies_after_reject = non_ess_after

        _, trackers_after = read_tracker_domains_from_perf_log(driver, domain)
        audit.trackers_after_reject = len(trackers_after)
        audit.known_tracker_domains_after_reject = sorted(trackers_after)

    # --- 7.6 Politique de confidentialité -----------------------------
    driver.switch_to.default_content()
    policy_link = find_policy_link(driver, url)
    audit.privacy_policy_link_found = policy_link is not None

    # --- 7.7 robots.txt -------------------------------------------------
    try:
        driver.get(f"{urlparse(url).scheme}://{domain}/robots.txt")
        time.sleep(1)
        audit.robots_txt_present = "user-agent" in driver.page_source.lower()
    except WebDriverException:
        pass

    return audit


# 8. Run

In [21]:
def run_audit(sites=SITES, output_csv="cookie_compliance_features.csv"):
    driver = build_driver()
    results = []
    try:
        for site in sites:
            print(f"[*] Analyse de {site} ...")
            audit = analyze_site(driver, site)
            results.append(audit.__dict__)
    finally:
        driver.quit()

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\n[+] Features enregistrées dans {output_csv}\n")
    cols = ["url", "reachable", "has_cookie_banner", "has_reject_button",
            "trackers_after_reject"]
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    return df


if __name__ == "__main__":
    run_audit()


[*] Analyse de https://www.lemonde.fr ...


[*] Analyse de https://www.leboncoin.fr ...



[+] Features enregistrées dans cookie_compliance_features.csv

                     url  reachable  has_cookie_banner  has_reject_button  trackers_after_reject
  https://www.lemonde.fr       True               True              False                      0
https://www.leboncoin.fr       True              False              False                      0
